#Goggle drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Import

In [ ]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import zipfile

#Set file path

In [ ]:
BASE_DIR = '/content/drive/MyDrive/kobert_model'
MODEL_ZIP_PATH = os.path.join(BASE_DIR, 'model_output.zip')
EXTRACTED_MODEL_DIR = os.path.join(BASE_DIR, 'model_output')

In [ ]:
if not os.path.exists(EXTRACTED_MODEL_DIR):
    try:
        with zipfile.ZipFile(MODEL_ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(BASE_DIR)
        print(f"'{MODEL_ZIP_PATH}' 파일이 '{BASE_DIR}'에 성공적으로 압축 해제되었습니다.")
    except FileNotFoundError:
        print(f"오류: '{MODEL_ZIP_PATH}' 파일을 찾을 수 없습니다. Google Drive 경로를 확인해주세요.")
    except Exception as e:
        print(f"압축 해제 중 오류 발생: {e}")
else:
    print(f"'{EXTRACTED_MODEL_DIR}' 폴더가 이미 존재합니다.")

'/content/drive/MyDrive/kobert_model/model_output' 폴더가 이미 존재합니다.


In [ ]:
LOAD_MODEL_PATH = os.path.join(EXTRACTED_MODEL_DIR, "final_model_kobert_sentiment")

#Import Tokenization infor and model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LOAD_MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(LOAD_MODEL_PATH)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

#Predictive news

In [ ]:
news_texts_to_predict = [
    '코스피, 1% 가까이 올라 3220선 회복',
    "'애플 호재'에 7만전자 복귀",
    "IPO 시장, '의무보유확약 확대'에 3개월 이상 투자금 증가",
    'BTS 돌아온 하이브, 2분기 매출 7천57억원 영업이익 659억원',
    "'테슬라에 애플까지' 파운드리 수주, '7만전자' 회복",
    "'최장 주행' 루시드 전기차, 일등공신은 '삼성SDI' 배터리",
    "'AI 대장주' 엔비디아, 시총 4조 달러 넘었다…전 세계 기업 최초",
    'AI, 방산, 원전 등의 신산업이 증시에서 주도적인 역할 수행',
    "엔비디아 '블랙웰' AI 반도체 수요가 공급 웃돌 가능성, 트럼프 정책에 수혜",
    "'부진한 2분기…실적 상향 종목 살펴야'",
    '실적 서프라이즈에 주가 큰 폭으로 상승하는 경우 많아',
    '기업 실적 개선, 주주환원 정책 강화',
    '반도체 업황 개선 기대',
    '글로벌 경제 회복 신호',
    '기관 매수세 유입으로 지수 상승',
    '외국인 투자자 순매수 증가',
    '신규 수주 공시, 역대 최대 규모',
    '기술 개발 성공 소식에 미래 성장 기대',
    '배당 확대 발표에 주주 가치 제고',
    '주요 제품 판매량 증가로 매출 성장',
    '삼성전자 실적 지금이 바닥… 반도체株 중 가장 유망',
    '정부의 반도체 산업 육성, 삼성전자 주가 우상향?',
    'AI 기업에 대한 투자 열풍에 주가 상승',
    '애플, 140조원 추가 투자 소식에 주가 5.9% 상승',
    'MS, AI 기술 접목으로 주가 28% 상승',
    '반도체주 기관계 자금 유입 지속',
    'HBM 기술력 확보에 한미반도체 주가 상승세',
    '바이오엔테크, 2분기 매출 2배 증가…주가 2% 상승',
    '리튬 반등·ESS 기대감에 이차전지주 급등',
    '이차전지 관련주, 리튬 가격 상승에 일제히 상승',
    '현대차, 신형 전기차 판매 호조에 주가 상승',
    '항공주, 2천억 자본 확충 결정으로 주가 안정화 기대',
    '화장품주, 2분기 호실적에 동반 강세',
    '신규 사업 진출 기대감에 주가 급등',
    '대규모 수주 계약 체결, 미래 성장 동력 확보',
    '자사주 매입 소각 결정, 주주가치 제고 효과',
    '새로운 치료제 개발 성공 소식에 바이오주 강세',
    '메타버스 플랫폼 출시, 시장 기대감 고조',
    '원자력 발전 정책 수혜주, 주가 상승세 지속',
    '태양광 에너지 기업, 정부 지원 확대에 주가 반등',
    '방산 기업, 해외 수출 계약 성사로 주가 고공행진',
    '로봇 관련주, 인공지능 기술 접목으로 주가 상승 탄력',
    'OTT 기업, 유료 구독자 증가로 실적 개선 기대',
    '게임주, 신작 출시 흥행에 주가 상승',
    '식품 기업, 해외 시장 진출 성공으로 매출 증가',
    '친환경 정책 수혜, 폐기물 처리 기업 주가 상승',
    '헬스케어 기업, 비대면 진료 확대 수혜로 주가 상승',
    '가전제품 기업, 프리미엄 제품 판매량 급증',
    '건설주, 재건축 사업 수주 기대감에 주가 상승',
    '해운주, 운임 지수 상승에 실적 개선 전망',
    "양대 악재 속...미국증시 '하락', 빅테크주 '뚝'",
    "'대주주 올가미 쓰기 싫다'…외국인·기관 탈출한 코스피, 3%대 폭락",
    "'중국 악재' 테슬라, 주가 8% 폭락 시총 1조 달러 붕괴",
    "테슬라, 이틀 내리 주가 급락…악재 연발 속 목표주가 '또' 하향 조정",
    "'대형 악재 겹친' 뉴욕증시 3대 지수 하락",
    "코스피, 관세 발효에 0.92% 하락… 반도체주 '뚝'",
    "증세 우려와 관세협상 여파에 주가에 악재로 작용",
    "기업 실적이 예상치에 훨씬 미치지 못하는 '어닝쇼크'",
    '부품 부족으로 공장 생산 중단',
    '글로벌 공급망 불안정',
    '경쟁 심화로 시장 점유율 감소',
    '금리 인상 가능성에 투자 심리 위축',
    '판매 가격 인하로 이익률 압박',
    '소비자 수요 둔화',
    '대규모 소송 리스크 발생',
    '정부 규제 강화 우려',
    '주요 원자재 가격 급등',
    '노동 쟁의 발생',
    '대규모 감원으로 인한 성장 둔화 우려',
    '예상보다 낮은 실적 발표에 투심 악화',
    '미국 증시, 소수의 AI 기업 빼면 뜨겁지 않다',
    "AI 거품 걷히면 '충격' 우려도",
    '슈퍼마이크로컴퓨터, 실적 미달로 주가 20% 이상 폭락',
    '반도체 업종, 네거티브 뉴스 증가로 주가 변동성 확대',
    "美 반도체주, 무역 휴전에도 전망 '부정적'",
    '반도체 매출 부진 예상에 주가 하락',
    '테슬라, 실적 미달로 8% 이상 폭락',
    '전기차 인센티브 폐지, 향후 경영 어려울 수 있어',
    '테슬라발 악재에 국내 이차전지주 투심 약화',
    "네이처셀, '조인트스템' 품목 허가 반려 소식에 하한가",
    'SK바이오사이언스 등 일부 바이오주 하락 마감',
    '금리인상, 금융회사에 큰 부담으로 작용',
    '미국 증시, 무역 전쟁 불확실성으로 하락',
    '해외 증시 급락에 국내 증시도 동반 하락',
    '기업 지배구조 리스크 부각에 주가 하락',
    '투자 경고 종목 지정, 주가 급락 주의보',
    '회사 경영진 횡령 배임 혐의 발생, 주가에 치명타',
    '신규 사업 철수 결정에 주가 급락',
    '대규모 유상증자 발표, 주가 희석 우려',
    '전환사채(CB) 대량 매도, 주가에 부담으로 작용',
    '주요 계약 해지 소식에 주가 하락',
    '환경 규제 강화로 인한 생산 비용 증가',
    '제품 리콜 사태 발생, 기업 이미지 타격',
    '글로벌 경기 침체 우려에 수출 기업 주가 약세',
    '환율 급등, 외화 부채 많은 기업 주가에 악영향',
    '중국 경제 성장 둔화, 관련주 주가 하락',
    '대외 불확실성 증폭, 투자심리 악화',
    '공급 과잉 문제로 인한 판매 단가 하락',
    '신기술 경쟁에서 뒤처지며 미래 성장 동력 상실',
    '주요 임원 사임 소식에 경영 불안정성 우려'
]


#Softmax [ Visualization of Output Values ]

**Use a function called softmax because the sum of the probability distributions is 1**

In [ ]:
from torch.nn.functional import softmax, sigmoid
PREDICTION_BATCH_SIZE = 32

#Derive results after prediction

In [ ]:
def predict_sentiment(texts, model, tokenizer, device, batch_size=PREDICTION_BATCH_SIZE):
    if not texts:
        print("Err: The text to predict is empty.")
        return [], []

    predictions = []
    probabilities = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]


        inputs = tokenizer(batch_texts,
                           truncation=True,
                           padding=True,
                           max_length=512,
                           return_tensors='pt').to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        batch_probs = softmax(logits, dim=1).cpu().numpy()
        batch_preds = torch.argmax(logits, dim=1).cpu().numpy()

        predictions.extend(batch_preds)
        probabilities.extend(batch_probs)

    return predictions, probabilities

In [ ]:
if news_texts_to_predict:
    predicted_labels, predicted_probabilities = predict_sentiment(
        news_texts_to_predict, model, tokenizer, device
    )

    results_df = pd.DataFrame({
        'Original News': news_texts_to_predict,
        'Predict label': predicted_labels,
        'Neg Probability (0)': [f"{p[0]:.3f}" for p in predicted_probabilities],
        'Pos Probability (1)': [f"{p[1]:.3f}" for p in predicted_probabilities]
    })

    results_df['Predictive sensitivity'] = results_df['Predict label'].map({0: 'Neg', 1: 'Pos'})

    print(results_df)

    PREDICTION_OUTPUT_DIR = os.path.dirname('/content/drive/MyDrive/result')
    PREDICTION_RESULTS_FILE = os.path.join(PREDICTION_OUTPUT_DIR, 'prediction_results.csv')

    try:
        results_df.to_csv(PREDICTION_RESULTS_FILE, index=False, encoding='utf-8-sig')
    except Exception as e:
        print(f"Err: {e}")

else:
    print("예측할 텍스트가 없으므로 예측을 수행하지 않습니다.")

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


                            Original News  Predict label Neg Probability (0)  \
0                 코스피, 1% 가까이 올라 3220선 회복              1               0.158   
1                        '애플 호재'에 7만전자 복귀              1               0.088   
2      IPO 시장, '의무보유확약 확대'에 3개월 이상 투자금 증가              1               0.004   
3   BTS 돌아온 하이브, 2분기 매출 7천57억원 영업이익 659억원              1               0.144   
4          '테슬라에 애플까지' 파운드리 수주, '7만전자' 회복              1               0.056   
..                                    ...            ...                 ...   
95                 중국 경제 성장 둔화, 관련주 주가 하락              0               0.596   
96                    대외 불확실성 증폭, 투자심리 악화              0               0.703   
97                  공급 과잉 문제로 인한 판매 단가 하락              0               0.638   
98              신기술 경쟁에서 뒤처지며 미래 성장 동력 상실              0               0.549   
99                주요 임원 사임 소식에 경영 불안정성 우려              0               0.528   

   Pos Probability (1) Predictive sensi

In [ ]:
path = '/content/drive/MyDrive/prediction_results.csv'

In [ ]:
data_df = pd.read_csv(path)
data_df.head()

,Original News,Predict label,Neg Probability (0),Pos Probability (1),Predictive sensitivity
0,"코스피, 1% 가까이 올라 3220선 회복",1,0.158,0.842,Pos
1,'애플 호재'에 7만전자 복귀,1,0.088,0.912,Pos
2,"IPO 시장, '의무보유확약 확대'에 3개월 이상 투자금 증가",1,0.004,0.996,Pos
3,"BTS 돌아온 하이브, 2분기 매출 7천57억원 영업이익 659억원",1,0.144,0.856,Pos
4,"'테슬라에 애플까지' 파운드리 수주, '7만전자' 회복",1,0.056,0.944,Pos


직접 긍정 뉴스기사 50, 부정뉴스기사 50개를 넣어 5번 확인해본 결과 => 약 91% 정도의 정확도를 보여준걸 확인